# 09 — RDD & Observability

The in-memory RDD API, plus observability: `explain`, `lineage`, `to_sql`, and session metrics.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. RDD basics

`parallelize`, `map`, `filter`, `reduce`, `collect`.

In [ ]:
sc = session.sparkContext
rdd = sc.parallelize([1, 2, 3, 4, 5])
print("collect:", rdd.collect())
print("map x2:", rdd.map(lambda x: x * 2).collect())
print("filter > 2:", rdd.filter(lambda x: x > 2).collect())
print("reduce sum:", rdd.reduce(lambda a, b: a + b))

## 2. RDD → DataFrame

`rdd.toDF()` bridges to a DataFrame.

In [ ]:
df = rdd.toDF(["value"])
df.show()

## 3. `df.explain()`

Logical plan + IRIS explain plan.

In [ ]:
vendas = session.table("vendas")
vendas.filter("estado = 'SP'").groupBy("cidade").count().explain()

## 4. `df.lineage()`

Transformation history.

In [ ]:
vendas.filter("estado = 'SP'").select("cidade", "valor").lineage(show=True)

## 5. `df.to_sql()`

Generated SQL.

In [ ]:
print(vendas.filter("estado = 'SP'").groupBy("cidade").count().to_sql())

## 6. Session observability metrics

Opt-in per-query metrics.

In [ ]:
session.config("irispark.observability", True)
session.sql("SELECT COUNT(*) FROM vendas")
metrics = getattr(session, "_metrics", [])
print("metrics recorded:", len(metrics))
for m in metrics[-3:]:
    print(m)
session.config("irispark.observability", False)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")